In [ ]:
# PROJECT - Heatshield san jose
# AI-powered heat-risk mapping and resilient city design

In [ ]:
╔══════════════════════════════════════════════╗
║       HEATSHIELD SAN JOSÉ                    ║
║   AI-Powered Urban Heat Resilience           ║
╠══════════════════════════════════════════════╣
║ 🔥 Max Temp     │ 🌡 Avg Temp │ ⚠ Hotspots  ║
║ 102.4°F         │ 94.1°F      │ 37           ║
╠══════════════════════════════════════════════╣
║                                              ║
║          INTERACTIVE HEAT RISK MAP           ║
║                                              ║
╠══════════════════════════════════════════════╣
║ CRITICAL ZONES                               ║
║                                              ║
║ Zone A → Shade + Trees                       ║
║ Zone B → Cool Pavement                       ║
║ Zone C → Cooling Corridor                    ║
╚══════════════════════════════════════════════╝

In [2]:
!pip -q install requests pandas numpy geopandas shapely folium matplotlib seaborn

In [3]:
import os
import time
import json
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

from shapely.geometry import shape
from IPython.display import display

In [4]:
from google.colab import userdata

FORTYGUARD_API_KEY = userdata.get("FORTYGUARD_API_KEY")

if not FORTYGUARD_API_KEY:
    raise ValueError("FortyGuard API key not found.")

print("✅ FortyGuard API key loaded successfully.")

✅ FortyGuard API key loaded successfully.


In [5]:
SAN_JOSE_AOI = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "name": "San Jose Study Area"
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [-121.950, 37.300],
                    [-121.850, 37.300],
                    [-121.850, 37.380],
                    [-121.950, 37.380],
                    [-121.950, 37.300]
                ]]
            }
        }
    ]
}

In [6]:
import requests
import json

url = "https://api.fortyguard.com/v1/heatmap"

headers = {
    "api-key": FORTYGUARD_API_KEY,
    "Content-Type": "application/json"
}

payload = {
    "polygon_aoi": {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [-121.90, 37.32],
                        [-121.87, 37.32],
                        [-121.87, 37.35],
                        [-121.90, 37.35],
                        [-121.90, 37.32]
                    ]]
                }
            }
        ]
    },
    "date_time": {
        "start_date": "2026-08-29",
        "start_time": "14:00",
        "filter_type": 1
    },
    "granularity": 100
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60
)

print("STATUS:", response.status_code)
print(json.dumps(response.json(), indent=2))

STATUS: 200
{
  "error": false,
  "status_code": 200,
  "message": "Heatmap Submitted Successfully",
  "data": {
    "activity_id": "5f58aced-27a4-4058-8d63-9952777a8c53"
  }
}


In [7]:
def submit_heatmap(aoi, date, time_str, granularity=100):

    url = f"{BASE_URL}/heatmap"

    payload = {
        "polygon_aoi": aoi,
        "date_time": {
            "start_date": date,
            "start_time": time_str,
            "filter_type": 1
        },
        "granularity": granularity
    }

    response = requests.post(
        url,
        headers=HEADERS,
        json=payload,
        timeout=60
    )

    print("HTTP Status:", response.status_code)

    if response.status_code != 200:
        print("API Error:")
        print(response.text)
        response.raise_for_status()

    result = response.json()

    activity_id = result["data"]["activity_id"]

    print("🔥 Heatmap submitted successfully!")
    print("Activity ID:", activity_id)

    return activity_id

In [8]:
import requests
import json
import time

from google.colab import userdata

API_KEY = userdata.get("FORTYGUARD_API_KEY")

BASE_URL = "https://api.fortyguard.com/v1"

HEADERS = {
    "api-key": API_KEY,
    "Content-Type": "application/json"
}

print("API key loaded:", bool(API_KEY))
print("BASE_URL:", BASE_URL)

API key loaded: True
BASE_URL: https://api.fortyguard.com/v1


In [9]:
def wait_for_activity(activity_id, max_wait=600, interval=5):

    status_url = f"https://api.fortyguard.com/v1/status/{activity_id}"

    start_time = time.time()

    while True:

        response = requests.get(
            status_url,
            headers={"api-key": FORTYGUARD_API_KEY},
            timeout=30
        )

        response.raise_for_status()

        data = response.json()["data"]
        status = data.get("status")

        print(f"Status: {status}")

        if status == "Completed":
            print("✅ Activity completed.")
            return data

        if status == "Failed":
            raise RuntimeError(
                f"FortyGuard activity failed: {activity_id}"
            )

        if time.time() - start_time > max_wait:
            raise TimeoutError(
                "FortyGuard activity exceeded maximum wait time."
            )

        time.sleep(interval)

In [10]:
activity_id = submit_heatmap(
    SAN_JOSE_AOI,
    "2026-08-29",
    "14:00",
    100
)

HTTP Status: 200
🔥 Heatmap submitted successfully!
Activity ID: 7faf67a8-f787-4aba-a61f-1651e66abe8e


In [11]:
result = wait_for_activity(activity_id)

print(json.dumps(result, indent=2)[:5000])

Status: Processing
Status: Processing
Status: Processing
Status: Processing
Status: Completed
✅ Activity completed.
{
  "activity_id": "7faf67a8-f787-4aba-a61f-1651e66abe8e",
  "status": "Completed",
  "result": {
    "map_data": {
      "type": "FeatureCollection",
      "features": [
        {
          "id": "0",
          "type": "Feature",
          "properties": {
            "tile_id": 0,
            "average_temperature": 27.5053,
            "min_temperature": 27.5053,
            "max_temperature": 27.5053
          },
          "geometry": {
            "type": "Polygon",
            "coordinates": [
              [
                [
                  -121.94978747013008,
                  37.3008470957796
                ],
                [
                  -121.94865489900094,
                  37.30083703932493
                ],
                [
                  -121.94864241296693,
                  37.30173392978465
                ],
                [
            

In [12]:
# Extract the completed heatmap result
heatmap_result = result

# Handle API response wrapper if present
if "data" in heatmap_result:
    heatmap_result = heatmap_result["data"]

if "result" in heatmap_result:
    heatmap_result = heatmap_result["result"]

map_data = heatmap_result.get("map_data", {})
stats_data = heatmap_result.get("stats_data", {})

print("✅ Map data loaded")
print("✅ Statistics data loaded")

print("Map data type:", type(map_data))
print("Statistics keys:", list(stats_data.keys()) if isinstance(stats_data, dict) else "N/A")

✅ Map data loaded
✅ Statistics data loaded
Map data type: <class 'dict'>
Statistics keys: ['temperature_stats', 'overall_temperature_distribution', 'normal_temperature_distribution', 'temperature_frequency']


In [13]:
temperature_stats = stats_data.get("temperature_stats", {})

print("🌡️ Temperature Statistics")
print("-" * 35)

print("Minimum Temperature:",
      temperature_stats.get("minimum"))

print("Maximum Temperature:",
      temperature_stats.get("maximum"))

print("Mean Temperature:",
      temperature_stats.get("mean"))

print("Median Temperature:",
      temperature_stats.get("median"))

🌡️ Temperature Statistics
-----------------------------------
Minimum Temperature: 24.5958
Maximum Temperature: 28.4254
Mean Temperature: 26.83234925296897
Median Temperature: None


In [14]:
features = map_data["features"]

records = []

for feature in features:

    properties = feature.get("properties", {})

    geometry = shape(feature["geometry"])

    records.append({
        "geometry": geometry,
        **properties
    })

heat_gdf = gpd.GeoDataFrame(
    records,
    crs="EPSG:4326"
)

print("Tiles:", len(heat_gdf))

display(heat_gdf.head())

Tiles: 7831


,geometry,tile_id,average_temperature,min_temperature,max_temperature
0,"POLYGON ((-121.94979 37.30085, -121.94865 37.3...",0,27.5053,27.5053,27.5053
1,"POLYGON ((-121.94865 37.30084, -121.94752 37.3...",1,27.4460,27.4460,27.4460
2,"POLYGON ((-121.94752 37.30083, -121.94639 37.3...",2,27.3900,27.3900,27.3900
3,"POLYGON ((-121.94639 37.30082, -121.94526 37.3...",3,27.3390,27.3390,27.3390
4,"POLYGON ((-121.94526 37.30081, -121.94412 37.3...",4,27.2948,27.2948,27.2948


In [16]:
# ============================================
# HEAT RISK ENGINE
# ============================================

# Use the API's actual temperature column
temp_col = "average_temperature"

# Convert to numeric safely
heat_gdf[temp_col] = pd.to_numeric(
    heat_gdf[temp_col],
    errors="coerce"
)

# Remove invalid temperature values
heat_gdf = heat_gdf.dropna(subset=[temp_col]).copy()

# Temperature range
t_min = heat_gdf[temp_col].min()
t_max = heat_gdf[temp_col].max()

# Avoid division by zero
if t_max == t_min:
    heat_gdf["temperature_score"] = 50.0
else:
    heat_gdf["temperature_score"] = (
        (heat_gdf[temp_col] - t_min) /
        (t_max - t_min)
    ) * 100

print("🔥 Heat Risk Engine Complete")
print(f"Minimum temperature: {t_min:.2f}°C")
print(f"Maximum temperature: {t_max:.2f}°C")

display(
    heat_gdf[
        ["tile_id", temp_col, "temperature_score"]
    ].head()
)

🔥 Heat Risk Engine Complete
Minimum temperature: 24.60°C
Maximum temperature: 28.43°C


,tile_id,average_temperature,temperature_score
0,0,27.5053,75.973992
1,1,27.4460,74.425527
2,2,27.3900,72.963234
3,3,27.3390,71.631502
4,4,27.2948,70.477334


In [18]:
# ============================================================
# HEAT RISK ENGINE
# ============================================================

import numpy as np
import pandas as pd

# Use the actual API column names
temp_col = "average_temperature"

# Temperature normalization
t_min = heat_gdf[temp_col].min()
t_max = heat_gdf[temp_col].max()

if t_max == t_min:
    heat_gdf["temperature_score"] = 0.0
else:
    heat_gdf["temperature_score"] = (
        (heat_gdf[temp_col] - t_min) / (t_max - t_min) * 100
    )

# ------------------------------------------------------------
# PERSISTENCE SCORE
# Since this heatmap is a spatial snapshot, use a stable
# baseline rather than pretending we have historical data.
# ------------------------------------------------------------

heat_gdf["persistence_score"] = 50.0

# ------------------------------------------------------------
# EXCEEDANCE SCORE
# Measures how far temperature is above a heat-risk threshold.
# Adjust threshold if the hackathon documentation specifies one.
# ------------------------------------------------------------

HEAT_THRESHOLD = 32.0

heat_gdf["exceedance_score"] = np.clip(
    (heat_gdf[temp_col] - HEAT_THRESHOLD) / 8.0 * 100,
    0,
    100
)

# ------------------------------------------------------------
# FINAL RISK SCORE
# ------------------------------------------------------------

heat_gdf["risk_score"] = (
    heat_gdf["temperature_score"] * 0.60 +
    heat_gdf["persistence_score"] * 0.25 +
    heat_gdf["exceedance_score"] * 0.15
)

heat_gdf["risk_score"] = heat_gdf["risk_score"].clip(0, 100)

print("✅ Heat Risk Engine completed")
print(f"Temperature range: {t_min:.2f}°C → {t_max:.2f}°C")
print(f"Average risk score: {heat_gdf['risk_score'].mean():.2f}")

✅ Heat Risk Engine completed
Temperature range: 24.60°C → 28.43°C
Average risk score: 47.54


In [19]:
def classify_heat(score):

    if score >= 80:
        return "CRITICAL"
    elif score >= 60:
        return "HIGH"
    elif score >= 40:
        return "MODERATE"
    else:
        return "LOW"


heat_gdf["risk_level"] = heat_gdf["risk_score"].apply(classify_heat)

print("✅ Hotspot classification completed")

display(
    heat_gdf[
        ["tile_id", "average_temperature", "risk_score", "risk_level"]
    ].head(10)
)

✅ Hotspot classification completed


,tile_id,average_temperature,risk_score,risk_level
0,0,27.5053,58.084395,MODERATE
1,1,27.4460,57.155316,MODERATE
2,2,27.3900,56.277940,MODERATE
3,3,27.3390,55.478901,MODERATE
4,4,27.2948,54.786401,MODERATE
5,5,27.2589,54.223940,MODERATE
6,6,27.2332,53.821287,MODERATE
7,7,27.2181,53.584709,MODERATE
8,8,27.2117,53.484437,MODERATE
9,9,27.2121,53.490704,MODERATE


In [20]:
# recommendation engine
def recommend_intervention(row):

    score = row["risk_score"]

    if score >= 80:
        return "Priority cooling + shade + trees + cool surfaces"
    elif score >= 60:
        return "Shade infrastructure + tree canopy"
    elif score >= 40:
        return "Increase vegetation + reflective surfaces"
    else:
        return "Maintain existing cooling infrastructure"


heat_gdf["recommendation"] = heat_gdf.apply(
    recommend_intervention,
    axis=1
)

print("✅ Recommendation engine completed")

display(
    heat_gdf[
        ["tile_id", "risk_score", "risk_level", "recommendation"]
    ].head(10)
)

✅ Recommendation engine completed


,tile_id,risk_score,risk_level,recommendation
0,0,58.084395,MODERATE,Increase vegetation + reflective surfaces
1,1,57.155316,MODERATE,Increase vegetation + reflective surfaces
2,2,56.277940,MODERATE,Increase vegetation + reflective surfaces
3,3,55.478901,MODERATE,Increase vegetation + reflective surfaces
4,4,54.786401,MODERATE,Increase vegetation + reflective surfaces
5,5,54.223940,MODERATE,Increase vegetation + reflective surfaces
6,6,53.821287,MODERATE,Increase vegetation + reflective surfaces
7,7,53.584709,MODERATE,Increase vegetation + reflective surfaces
8,8,53.484437,MODERATE,Increase vegetation + reflective surfaces
9,9,53.490704,MODERATE,Increase vegetation + reflective surfaces


In [21]:
# final data validation
print("Rows:", len(heat_gdf))
print("Columns:", list(heat_gdf.columns))
print(heat_gdf["risk_level"].value_counts())
print(heat_gdf["risk_score"].describe())

Rows: 7831
Columns: ['geometry', 'tile_id', 'average_temperature', 'min_temperature', 'max_temperature', 'temperature_score', 'persistence_score', 'exceedance_score', 'risk_score', 'risk_level', 'recommendation']
risk_level
MODERATE    4817
LOW         2019
HIGH         995
Name: count, dtype: int64
count    7831.000000
mean       47.540985
std        12.008745
min        12.500000
25%        39.928191
50%        49.446940
75%        55.169469
max        72.500000
Name: risk_score, dtype: float64


In [23]:
# interactive map
import folium
from folium.features import GeoJsonTooltip

# Make sure coordinates are WGS84
heat_map_gdf = heat_gdf.to_crs(epsg=4326)

# Create map using OpenStreetMap — NO API KEY REQUIRED
m = folium.Map(
    location=[37.3382, -121.8863],
    zoom_start=11,
    tiles="OpenStreetMap",
    control_scale=True
)

risk_colors = {
    "CRITICAL": "#d73027",
    "HIGH": "#fc8d59",
    "MODERATE": "#fee08b",
    "LOW": "#91cf60"
}

def style_feature(feature):
    level = feature["properties"].get("risk_level", "LOW")

    return {
        "fillColor": risk_colors.get(level, "#91cf60"),
        "color": "#333333",
        "weight": 0.5,
        "fillOpacity": 0.65
    }

folium.GeoJson(
    heat_map_gdf.to_json(),
    name="Heat Risk Zones",
    style_function=style_feature,
    tooltip=GeoJsonTooltip(
        fields=[
            "tile_id",
            "average_temperature",
            "risk_score",
            "risk_level",
            "recommendation"
        ],
        aliases=[
            "Tile",
            "Temperature °C",
            "Risk Score",
            "Risk Level",
            "Recommendation"
        ],
        localize=True,
        sticky=False
    )
).add_to(m)

folium.LayerControl().add_to(m)

# Automatically fit map to your actual heatmap data
bounds = heat_map_gdf.total_bounds
m.fit_bounds([
    [bounds[1], bounds[0]],
    [bounds[3], bounds[2]]
])

m

In [25]:
# KPI dashboard
from IPython.display import display, HTML

critical = (heat_gdf["risk_level"] == "CRITICAL").sum()
high = (heat_gdf["risk_level"] == "HIGH").sum()
moderate = (heat_gdf["risk_level"] == "MODERATE").sum()
low = (heat_gdf["risk_level"] == "LOW").sum()

avg_temp = heat_gdf["average_temperature"].mean()
max_temp = heat_gdf["max_temperature"].max()
avg_risk = heat_gdf["risk_score"].mean()

display(HTML(f"""
<style>
.kpi-container {{
    display: flex;
    gap: 18px;
    margin: 20px 0;
    font-family: Arial, sans-serif;
    flex-wrap: wrap;
}}

.kpi-card {{
    width: 210px;
    padding: 20px;
    border-radius: 14px;
    background: white;
    border: 1px solid #d9dee5;
    box-shadow: 0 4px 12px rgba(0,0,0,0.10);
}}

.kpi-title {{
    font-size: 15px;
    font-weight: 600;
    color: #374151;
    margin-bottom: 10px;
}}

.kpi-value {{
    font-size: 30px;
    font-weight: 700;
    color: #111827;
}}

.kpi-sub {{
    font-size: 12px;
    color: #6b7280;
    margin-top: 5px;
}}
</style>

<div class="kpi-container">

<div class="kpi-card">
<div class="kpi-title">🌡️ Average Temperature</div>
<div class="kpi-value">{avg_temp:.1f}°C</div>
<div class="kpi-sub">Across all monitored zones</div>
</div>

<div class="kpi-card">
<div class="kpi-title">🔥 Maximum Temperature</div>
<div class="kpi-value">{max_temp:.1f}°C</div>
<div class="kpi-sub">Highest detected value</div>
</div>

<div class="kpi-card">
<div class="kpi-title">⚠️ Average Risk</div>
<div class="kpi-value">{avg_risk:.1f}/100</div>
<div class="kpi-sub">Overall heat-risk score</div>
</div>

<div class="kpi-card">
<div class="kpi-title">🚨 Critical Zones</div>
<div class="kpi-value">{critical}</div>
<div class="kpi-sub">Require immediate attention</div>
</div>

<div class="kpi-card">
<div class="kpi-title">🔴 High-Risk Zones</div>
<div class="kpi-value">{high}</div>
<div class="kpi-sub">Priority intervention areas</div>
</div>

</div>
"""))

In [26]:
# hotspot ranking
hotspots = (
    heat_gdf[
        heat_gdf["risk_level"].isin(["CRITICAL", "HIGH"])
    ]
    .sort_values("risk_score", ascending=False)
    [["tile_id", "average_temperature", "risk_score",
      "risk_level", "recommendation"]]
)

display(hotspots.head(15))

,tile_id,average_temperature,risk_score,risk_level,recommendation
1143,1143,28.4254,72.500000,HIGH,Shade infrastructure + tree canopy
1055,1055,28.4112,72.277522,HIGH,Shade infrastructure + tree canopy
1231,1231,28.4056,72.189785,HIGH,Shade infrastructure + tree canopy
1142,1142,28.3805,71.796532,HIGH,Shade infrastructure + tree canopy
1054,1054,28.3754,71.716628,HIGH,Shade infrastructure + tree canopy
967,967,28.3740,71.694694,HIGH,Shade infrastructure + tree canopy
1230,1230,28.3543,71.386046,HIGH,Shade infrastructure + tree canopy
966,966,28.3485,71.295174,HIGH,Shade infrastructure + tree canopy
1053,1053,28.3450,71.240338,HIGH,Shade infrastructure + tree canopy
1141,1141,28.3424,71.199603,HIGH,Shade infrastructure + tree canopy


In [27]:
heat_gdf.to_file(
    "HeatShield_SanJose_Final.geojson",
    driver="GeoJSON"
)

heat_gdf.drop(columns="geometry").to_csv(
    "HeatShield_SanJose_Risk_Report.csv",
    index=False
)

print("✅ Final files exported.")

✅ Final files exported.
